In [1]:
import pandas as pd 
import numpy as np
from scipy import stats

In [ ]:
# Importazione dataset test-pilota
pilot = pd.read_csv(r'Project\03. Variabilità di Peso nel Confezionamento Caffè\Dataset\Clean\pilot_clean.csv')

# Dataset

In [3]:
# Descrizione dataset
pilot.describe()

,weight_grams
count,279.000000
mean,249.196989
std,2.238542
min,242.470000
25%,248.200000
50%,249.340000
75%,250.550000
max,255.400000


In [4]:
pilot.info()

<class 'pandas.DataFrame'>
RangeIndex: 279 entries, 0 to 278
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          279 non-null    str    
 1   shift         279 non-null    str    
 2   machine       279 non-null    str    
 3   batch_id      279 non-null    str    
 4   weight_grams  279 non-null    float64
dtypes: float64(1), str(4)
memory usage: 22.6 KB


In [5]:
pilot.head()

,date,shift,machine,batch_id,weight_grams
0,2026-06-08,Notte,Macchina_2_Riferimento,P2015,249.80
1,2026-06-01,Notte,Macchina_1_Contromisura,P2000,252.29
2,2026-06-06,Notte,Macchina_1_Contromisura,P2010,249.93
3,2026-06-01,Notte,Macchina_2_Riferimento,P2001,246.46
4,2026-06-11,Notte,Macchina_2_Riferimento,P2021,250.02


Dopo aver verificato la presenza di valori nulli nel dataset procedo con l'analisi statistica.

# t-test di Welch a una coda
Questo test determina se ci sono differenze significative tra le medie dei due gruppi, utilizzando il test ad una coda possiamo effettivamente valutare se il peso medio è superiore rispetto al riferimento

- $H_0$: il peso medio è minore o uguale al riferimento (μ₁≤μ₂)
- $H_1$: il peso medio è maggiore al al riferimento (μ₁>μ₂)
- $\alpha$: 0,05

In [6]:
statistiche = pilot.groupby('machine').agg(
    letture = ('weight_grams','size'),
    media_pesate = ('weight_grams', 'mean'),
    dev_std = ('weight_grams', 'std')
)

In [7]:
riferimento = pilot[pilot['machine']=='Macchina_2_Riferimento']
contromisura = pilot[pilot['machine']=='Macchina_1_Contromisura']
contromisura

,date,shift,machine,batch_id,weight_grams
1,2026-06-01,Notte,Macchina_1_Contromisura,P2000,252.29
2,2026-06-06,Notte,Macchina_1_Contromisura,P2010,249.93
5,2026-06-05,Notte,Macchina_1_Contromisura,P2008,250.54
6,2026-06-11,Notte,Macchina_1_Contromisura,P2020,250.29
7,2026-06-04,Notte,Macchina_1_Contromisura,P2006,250.67
...,...,...,...,...,...
268,2026-06-03,Notte,Macchina_1_Contromisura,P2004,250.22
272,2026-06-09,Notte,Macchina_1_Contromisura,P2016,251.28
273,2026-06-14,Notte,Macchina_1_Contromisura,P2026,249.50
275,2026-06-02,Notte,Macchina_1_Contromisura,P2002,247.50


In [8]:
# Test Welch
t_stat, p_value = stats.ttest_ind(
    contromisura['weight_grams'],
    riferimento['weight_grams'],
    equal_var=False   
)

In [9]:
print(t_stat, p_value)

3.6210304029783007 0.00037104871555408947


In [10]:
p_one_tail = p_value / 2
print(p_one_tail)

0.00018552435777704473


## Risultato
Il valore di p-value ad una coda (0.000186) è molto inferiore al livello di significatività ($\alpha$). Questo conferma che l'ipotesi nulla ($H_0$) è rigettata 

# Test Levene
Questo test è stato effettuato per determinare una differenza significativa tra le varianze dei gruppi analizzati. 

- $H_0$: le varianze dei due gruppi sono uguali 
- $H_1$: le varianze dei due gruppi hanno una differenza significativa 
- $\alpha$: Intervallo di significatività (0,05)


In [11]:
# Test Levene
stat, p_value_levene = stats.levene(
    contromisura['weight_grams'],
    riferimento['weight_grams']
)

In [12]:
print(stat, p_value_levene)

49.81378418684147 1.3594595056805263e-11


## Risultati
la direzione — quale macchina ha varianza minore — è confermata dai descrittivi: Macchina_1 std=1,358 vs Macchina_2 std=2,782, non dal test di Levene in sé, che conferma solo la significatività della differenza

# Statistiche 
Per verificare se le modifiche apportate hanno migliorato la capability della macchina, analizzo i dati delle due macchine (Contromisura e Riferimento) per ricavare le statistiche utili: 
- Media pesata
- Deviazione standard 
- Cpk
- UCL
- LCL

In [13]:
lsl = 245
usl = 255
Cpk_target = 1.33

statistiche['cpk'] = np.minimum((statistiche['media_pesate']-lsl)/(3*statistiche['dev_std']), (usl-statistiche['media_pesate'])/(3*statistiche['dev_std']))
statistiche ['ucl'] = statistiche['media_pesate']+(3*statistiche['dev_std'])
statistiche ['lcl'] = statistiche['media_pesate']-(3*statistiche['dev_std'])
statistiche

,letture,media_pesate,dev_std,cpk,ucl,lcl
machine,,,,,,
Macchina_1_Contromisura,139,249.672734,1.357939,1.147016,253.746551,245.598917
Macchina_2_Riferimento,140,248.724643,2.782150,0.446255,257.071093,240.378193


la contromisura ha prodotto un miglioramento **statisticamente significativo** sia sulla media che sulla varianza (confermato da Welch e Levene), quasi triplicando il Cpk (da 0,446 a 1,147). Il miglioramento **non è ancora sufficiente** a raggiungere il target di processo (Cpk ≥ 1,33). Il margine tra il nuovo LCL (245,60g) e la specifica cliente (LSL 245g) resta sottile (~0,6g), a conferma che il processo, pur migliorato, opera ancora vicino al limite.

# Business problem
Sulla base delle modifiche e delle contromisure effettuate valutiamo il ritorno economico per l'azienda in modo da quantificare i guadagni implementando le contromisure ipotizzate 

| Voce | Valore |
|---|---:|
| Margine di contribuzione / confezione | €2,50 |
| Produzione turno Notte | 3.000 confezioni/notte |
| Giorni operativi/anno | ~240 (dichiarare valore finale scelto) |
| Costo per reclamo gestito | €150 |
| Costo orario intervento | €25/h |
| Durata checklist | 15 min/notte |
| Tasso di conversione difetto→reclamo | ancorato al dato storico: ~90.000 confezioni sotto LSL/anno (stimate) ÷ ~18 reclami/anno storici ≈ **1 reclamo ogni 5.000 confezioni sotto peso** |

In [14]:
# Business problem
prezzo_vendita = 8
margine = 2.5
produzione_notte = 3000
giorni_lavorati = 300
produzione_annua_notte = produzione_notte * giorni_lavorati
costo_reclamo = 150
costo_operatore = 25
durata_checklist = 15/60 

In [15]:
#Perdita % sulla produzione delle due macchine
sotto_contromisura = ((contromisura['weight_grams']<245).sum()/len(contromisura['weight_grams']))*100
sotto_riferimento = ((riferimento['weight_grams']<245).sum()/len(riferimento['weight_grams']))*100

# Perdita annuale stimata sulle due macchine 
annuale_contromisura = produzione_annua_notte * (sotto_contromisura/100)
annuale_riferimento = produzione_annua_notte * (sotto_riferimento/100)

# Confezioni recuperate con la contromisura
recupero = annuale_riferimento - annuale_contromisura

# Margine recuperato dalla perdita
margine_recuperato = recupero*margine

# Benerfici economici dei reclami evitati (stimo circa 1 reclamo ogni 500 prodotti)
reclami = recupero/500
beneficio_reclamo = reclami * costo_reclamo

# Costo operatore per manutenzione come da checklist
ore_annuali = giorni_lavorati * durata_checklist
costo_checklist = ore_annuali * costo_operatore

# Totale beneficio netto
beneficio_netto = margine_recuperato + beneficio_reclamo - costo_checklist

# ROI (Return of Investment)
ROI = (beneficio_netto/costo_checklist)*100

In [16]:
print(f"Perdita annua PRIMA (Macchina_2): {annuale_riferimento:.0f} confezioni")
print(f"Perdita annua DOPO (Macchina_1): {annuale_contromisura:.0f} confezioni")
print(f"Confezioni recuperate/anno: {recupero:.0f}")
print(f"Margine recuperato: €{margine_recuperato:.2f}")
print(f"Reclami evitati/anno: {reclami:.2f}")
print(f"Beneficio da reclami: €{beneficio_reclamo:.2f}")
print(f"Costo checklist/anno: €{costo_checklist:.2f}")
print(f"Beneficio netto: €{beneficio_netto:.2f}")
print(f"ROI: {ROI:.0f}%")

Perdita annua PRIMA (Macchina_2): 90000 confezioni
Perdita annua DOPO (Macchina_1): 0 confezioni
Confezioni recuperate/anno: 90000
Margine recuperato: €225000.00
Reclami evitati/anno: 180.00
Beneficio da reclami: €27000.00
Costo checklist/anno: €1875.00
Beneficio netto: €250125.00
ROI: 13340%


Il valore di perdita annua dopo la contromisura (0) risulta poco realistico. Questo perchè potrebbero esserci degli scarti di produzione dovuti ad altre cause oppure perchè durante l'avvio della linea potrebbero esserci degli scarti iniziali da considerare. 
Per rendere più realistico questo valore possiamo utilizzare la regola del successo dello zero: *quando in un campione non nessun evento, un limite superiore prudente per il vero tasso di difettosità*, con il 95% di confidenza,  è approssimativamente 
$$\text{tasso massimo plausibile} = \frac{3}{n}$$
dove $n = \text{numero di osservazioni}$

- I giorni lavorati risultano superiori a quelli reali. Considerando che un anno è composto da 52 settimane e che 2 giorni a settimana (sabato e domenica) la produzione è sospesa, dobbiamo sottrarre 100 giorni circa a 365 (giorni annuali). Se a questi sottraiamo anche le festività e i ponti di festa, possiamo stimare ragionevolmente circa 240 giorni lavorati in un anno 
- Nel caso dei reclami è stato visto che in 2 mesi ci sono stati 3 reclami (18 reclami anno) e quindi possiamo stimare una conversione più realistica anche nel tasso di reclami. Utilizziamo il numero di reclami stimato all'anno (18), ipotizziamo che provengano dal turno di notte (il turno con Cpk = 0.53, il più basso), infine definiamo che il riferimento produceva 90.000 confezioni in un anno e di queste ogni 18 veniva effettuato un reclamo. quindi 
$$ \text{tasso di conversione implicito} = \frac{\text{confezioni sotto LSL}}{\text{reclami osservati}} = \frac{90.000}{18} = {5.000}

In [17]:
# Nuovi parametri produzione
prezzo_vendita = 8
margine = 2.5
produzione_notte = 3000
giorni_lavorati_nuovo = 240
produzione_annua_notte = produzione_notte * giorni_lavorati_nuovo
costo_reclamo = 150
costo_operatore = 25
durata_checklist = 15/60 
storico_reclami_annuale = 18 
stima_reclami = 5000

In [18]:
#Perdita % sulla produzione delle due macchine
sotto_contromisura_stimato = (3 / len(contromisura['weight_grams']))*100
sotto_riferimento = ((riferimento['weight_grams']<245).sum()/len(riferimento['weight_grams']))*100

# Perdita annuale stimata sulle due macchine 
annuale_contromisura_stimato = produzione_annua_notte * (sotto_contromisura_stimato/100)
annuale_riferimento = produzione_annua_notte * (sotto_riferimento/100)

# Confezioni recuperate con la contromisura
recupero_stimato = annuale_riferimento - annuale_contromisura_stimato

# Margine recuperato dalla perdita
margine_recuperato_stimato = recupero_stimato*margine

# Benerfici economici dei reclami evitati
reclami = recupero_stimato/stima_reclami
beneficio_reclamo = reclami * costo_reclamo

# Costo operatore per manutenzione come da checklist
ore_annuali = giorni_lavorati_nuovo * durata_checklist
costo_checklist = ore_annuali * costo_operatore

# Totale beneficio netto
beneficio_netto = margine_recuperato_stimato + beneficio_reclamo - costo_checklist

# Calcolo dei giorni lavorativi necessari per ammortizzare la manutezione dell'intero anno
payback_giorni = (costo_checklist / beneficio_netto) * 365



In [19]:
print(f"Perdita annua PRIMA (Macchina_2): {annuale_riferimento:.0f} confezioni")
print(f"Perdita annua DOPO (Macchina_1): {annuale_contromisura_stimato:.0f} confezioni")
print(f"Confezioni recuperate/anno: {recupero:.0f}")
print(f"Margine recuperato: €{margine_recuperato:.2f}")
print(f"Reclami evitati/anno: {reclami:.2f}")
print(f"Beneficio da reclami: €{beneficio_reclamo:.2f}")
print(f"Costo checklist/anno: €{costo_checklist:.2f}")
print(f"Beneficio netto: €{beneficio_netto:.2f}")
print(f"Payback giorni: {payback_giorni:.0f}")

Perdita annua PRIMA (Macchina_2): 72000 confezioni
Perdita annua DOPO (Macchina_1): 15540 confezioni
Confezioni recuperate/anno: 90000
Margine recuperato: €225000.00
Reclami evitati/anno: 11.29
Beneficio da reclami: €1693.81
Costo checklist/anno: €1500.00
Beneficio netto: €141344.89
Payback giorni: 4


In [20]:
# ============================================================
# BUSINESS CASE — Versione finale con payback e sensitivity
# ============================================================

# --- Ricalcolo pulito dei valori base (evita variabili "stantie") ---
produzione_annua_notte = produzione_notte * giorni_lavorati_nuovo

annuale_riferimento = produzione_annua_notte * (sotto_riferimento / 100)
annuale_contromisura_stimato = produzione_annua_notte * (sotto_contromisura_stimato / 100)
recupero_stimato = annuale_riferimento - annuale_contromisura_stimato

margine_recuperato_stimato = recupero_stimato * margine
costo_checklist = giorni_lavorati_nuovo * durata_checklist * costo_operatore

print(f"Produzione annua turno Notte: {produzione_annua_notte:.0f} confezioni")
print(f"Confezioni sotto LSL PRIMA:   {annuale_riferimento:.0f}")
print(f"Confezioni sotto LSL DOPO:    {annuale_contromisura_stimato:.0f}")
print(f"Confezioni recuperate/anno:   {recupero_stimato:.0f}")
print(f"Margine recuperato:           €{margine_recuperato_stimato:,.2f}")
print(f"Costo checklist/anno:         €{costo_checklist:,.2f}")

# ============================================================
# ANALISI DI SENSITIVITÀ — tasso di conversione difetto → reclamo
# ============================================================
# Tre scenari, invece di un unico numero puntuale, perché il tasso
# 1/5.000 è stimato da soli 3 reclami osservati in 2 mesi — un
# campione troppo piccolo per un'unica cifra "certa".

scenari = {
    "Conservativo": 3000,   # più reclami stimati -> beneficio reclami più alto -> ROI più prudente NO:
                            # attenzione al verso: tasso PIÙ BASSO (es. 1/3000) = più reclami evitati
                            # quindi "conservativo" qui significa: assumo che il legame difetto->reclamo
                            # sia più forte di quanto suggerisce lo storico, quindi il beneficio è meno
                            # dipendente da un'assunzione ottimistica sul beneficio_reclamo (che pesa poco)
    "Base":          5000,  # ancorato al dato storico (90.000 sotto LSL / 18 reclami storici/anno)
    "Ottimistico":   8000,  # assume un legame difetto->reclamo più debole (meno reclami evitati)
}

righe = []
for nome, tasso in scenari.items():
    reclami_evitati = recupero_stimato / tasso
    beneficio_reclamo = reclami_evitati * costo_reclamo
    beneficio_netto = margine_recuperato_stimato + beneficio_reclamo - costo_checklist
    roi = (beneficio_netto / costo_checklist) * 100
    payback_giorni = (costo_checklist / beneficio_netto) * 365

    righe.append({
        "Scenario": nome,
        "Tasso (1 reclamo ogni N)": tasso,
        "Reclami evitati/anno": round(reclami_evitati, 1),
        "Beneficio reclami (€)": round(beneficio_reclamo, 2),
        "Beneficio netto (€)": round(beneficio_netto, 2),
        "ROI (%)": round(roi, 0),
        "Payback (giorni)": round(payback_giorni, 1),
    })

sensitivity_df = pd.DataFrame(righe)
sensitivity_df

Produzione annua turno Notte: 720000 confezioni
Confezioni sotto LSL PRIMA:   72000
Confezioni sotto LSL DOPO:    15540
Confezioni recuperate/anno:   56460
Margine recuperato:           €141,151.08
Costo checklist/anno:         €1,500.00


,Scenario,Tasso (1 reclamo ogni N),Reclami evitati/anno,Beneficio reclami (€),Beneficio netto (€),ROI (%),Payback (giorni)
0,Conservativo,3000,18.8,2823.02,142474.10,9498.0,3.8
1,Base,5000,11.3,1693.81,141344.89,9423.0,3.9
2,Ottimistico,8000,7.1,1058.63,140709.71,9381.0,3.9


**Interpretazione:** il beneficio economico è dominato dal recupero di margine su produzione, non dai reclami evitati — la conclusione economica è quindi robusta rispetto all'incertezza sul tasso di conversione reclami (variazione <1,5% tra scenari). Il ROI elevato è coerente con la natura dell'intervento: costo bassissimo (~€1.500/anno) su un processo ad alto volume con margine reale per unità — il **payback period (~4 giorni)** è la cifra più immediatamente comunicabile a un management non tecnico.